# Solving Your Business Problems Using Prompts and LLMs in SAP's Generative AI Hub (Facility Management Example)

Start with introducing the scenario:
- Company wants to analyse support mail
- See company description in `company-scope.md`
- Incoming mails should be assigned to the correct service category and assigned the correct urgency and sentiment

**Goal**: Analyse every message and extract:
- `urgency`
- `sentiment`
- `categories`

## Install packages

In [1]:
// npm install @sap-ai-sdk/ai-api @sap-ai-sdk/orchestration dotenv

In [ ]:
import dotenv from 'dotenv';
dotenv.config();

console.log(process.env.AICORE_SERVICE_KEY); 

## Imports and Settings

In [3]:
const EXAMPLE_MESSAGE_IDX = 10;

## Load Data

In [4]:
const filePath = "./filtered_mails-hardest.jsonl";
let mails = [];

try {
  const content = await Deno.readTextFile(filePath);
  mails = content
    .split("\n")
    .filter(line => line.trim())
    .map(line => JSON.parse(line));
  console.log("Loaded", mails.length, "mails");
} catch (error) {
  console.error("Error reading the file:", error);
}

console.log(mails.slice(0, 2)); 


Loaded 200 mails
[
  {
    id: "c36dbb22-9efb-482e-b9ae-1ede7afa0696",
    message: "Subject: Urgent Assistance Required for Facility Management and Safety Concerns\n" +
      "\n" +
      "Dear Support Team,\n" +
      "\n" +
      "I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.\n" +
      "\n" +
      "Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security p

In [5]:
// Split mails into dev_set and test_set
const midpoint = Math.floor(mails.length / 2);
const dev_set = mails.slice(0, midpoint);
const test_set = mails.slice(midpoint);

// Take first 20 from test_set
const test_set_small = test_set.slice(0, 20);

console.log("Dev set size:", dev_set.length);
console.log("Test set size:", test_set.length);
console.log("Small test set size:", test_set_small.length);


Dev set size: 100
Test set size: 100
Small test set size: 20


In [6]:
// Initialize sets
let categories = new Set();
let urgency = new Set();
let sentiment = new Set();

// Loop through mails
for (const mail of mails) {
  categories = new Set([...categories, ...mail.ground_truth.categories]);
  urgency.add(mail.ground_truth.urgency);
  sentiment.add(mail.ground_truth.sentiment);
}

// Convert sets into option_lists (formatted like Python f-strings with backticks)
const option_lists = {
  urgency: Array.from(urgency).map(entry => `\`${entry}\``).join(", "),
  sentiment: Array.from(sentiment).map(entry => `\`${entry}\``).join(", "),
  categories: Array.from(categories).map(entry => `\`${entry}\``).join(", "),
};

console.log(option_lists);


{
  urgency: "`high`, `medium`, `low`",
  sentiment: "`neutral`, `positive`, `negative`",
  categories: "`facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`"
}


In [7]:
option_lists

{
  urgency: "`high`, `medium`, `low`",
  sentiment: "`neutral`, `positive`, `negative`",
  categories: "`facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`"
}

## Create a Configuration

In [ ]:
import { ConfigurationApi } from '@sap-ai-sdk/ai-api';

async function createConfiguration() {
  const requestBody: ConfigurationBaseData = {
    name: 'orchestration-config',
    executableId: 'orchestration',
    scenarioId: 'orchestration',
    parameterBindings: [
          {
              "key": "modelFilterList", 
              "value": "null"  
          },
          {
              "key": "modelFilterListType",
              "value": "allow"
          }
      ],
    inputArtifactBindings: []
  };

  const responseData: ConfigurationCreationResponse =
    await ConfigurationApi.configurationCreate(requestBody, {
      'AI-Resource-Group': 'grounding'
    }).execute();
  return responseData;
}

// usage
const orchestrationConfig = await createConfiguration();
orchestrationConfig;


## Create a Deployment

In [ ]:
import { DeploymentApi } from "@sap-ai-sdk/ai-api";

async function createDeployment(configurationId: string) {
  const requestBody = { configurationId };
  const responseData = await DeploymentApi.deploymentCreate(requestBody, {
    "AI-Resource-Group": "grounding",
  }).execute();
  return responseData;
}

// ---- USAGE ----

const orchestrationConfig = { id: orchestrationConfig.id };

const deployment = await createDeployment(orchestrationConfig.id);
console.log("🚀 Deployment scheduled:", deployment.id);


In [ ]:
import { DeploymentApi } from "@sap-ai-sdk/ai-api";

// simple spinner chars
const spinnerChars = ["|", "/", "-", "\\"];

async function spinner(checkCallback, timeout = 300_000, checkEvery = 10_000) {
  const start = Date.now();
  let i = 0;

  while (Date.now() - start < timeout) {
    const result = await checkCallback();
    if (result) {
      return result;
    }

    process.stdout.write(
      `\rWaiting for the deployment to become ready... ${spinnerChars[i % spinnerChars.length]}`
    );
    i++;

    await new Promise((res) => setTimeout(res, checkEvery));
  }

  throw new Error("Deployment did not become ready in time");
}

export async function deployAndWait(configurationId, resourceGroup ) {
  // Step 1: Create deployment
  const deployment = await DeploymentApi.deploymentCreate(
    { configurationId },
    { "AI-Resource-Group": "grounding" }
  ).execute();

  console.log("🚀 Deployment scheduled:", deployment.id);

  // Step 2: Spinner with callback
  const readyDeployment = await spinner(async () => {
    try {
      const updated = await DeploymentApi.deploymentGet(deployment.id, {
        "AI-Resource-Group": resourceGroup,
      }).execute();

      return updated.status === "RUNNING" ? updated : null;
    } catch (err: any) {
      if (err.response?.status === 404) {
        return null; // not registered yet, keep waiting
      }
      throw err;
    }
  });

  console.log(`\n✅ Deployment is ready: ${readyDeployment.deploymentUrl}`);
  return readyDeployment;
}
//Usage
const deployment = await deployAndWait(orchestrationConfig.id);
console.log("🎯 Deployment URL:", deployment.deploymentUrl);


In [11]:
import { OrchestrationClient } from "@sap-ai-sdk/orchestration";

export async function sendRequest({
  prompt,
  model = "meta--llama3.1-70b-instruct",
  print = false,
  ...kwargs
}) {
  const client = new OrchestrationClient({
    llm: { model_name: model },
    templating: {
      template: prompt,  
    },
  });

  // run orchestration
  const answer = await client.chatCompletion({
    inputParams: kwargs,
  });

  // 🔹 reconstruct formatted prompt
  let formattedPrompt = "";
  if (answer.templating_result) {
    formattedPrompt = answer.templating_result
      .map(t => t.content) // ✅ only content, no roles
      .join("\n");
  } else {
    // fallback: local interpolation
    const interpolate = (template, values) =>
      template.replace(/{{\?(\w+)}}/g, (_, key) => values[key] || "");

    formattedPrompt = prompt
      .map(m => interpolate(m.content, kwargs))
      .join("\n");
  }

  if (print) {
    console.log("<-- PROMPT --->\n" + formattedPrompt);
    console.log("<--- RESPONSE --->\n" + answer.getContent());
  }

  return {
    result: answer.getContent(),
    formattedPrompt,
  };
}

## Sentiment & Urgency Simple

In [12]:
const mail = dev_set[EXAMPLE_MESSAGE_IDX];

In [13]:
// ✅ define the prompt
const prompt1 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Your task is to extract:
- urgency
- sentiment
Giving the following message:`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

const f1 = async (mail) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt1,
    input: mail.message,
  });

  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// usage
const response = await f1(mail);


<-- PROMPT --->
You are an intelligent assistant. Your task is to extract:
- urgency
- sentiment
Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential Complex Name], where I have been residing for the past few years. I have always appreciated the meticulous care and attention your team provides in maintaining our facilities.

However, I am currently facing a pressing issue with the HVAC system in my apartment. Over the past few days, the system has been malfunctioning, resulting in inconsistent temperatures and, at times, complete shutdowns. Given the current weather conditions, this has become quite unbearable and is affecting my daily routine significantly.

I have attempted to troubleshoot the problem by resetting the system and checking the thermostat settings, but these efforts have not yielded any improvement. The situation seems to be bey

## Sentiment & Urgency with Choices

In [14]:

// ✅ define the prompt
const prompt2 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Your task is to extract:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "category" as one of {{?categories}}

Giving the following message:`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

const f2 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt2,
    input: input,
    ...option_lists   // <-- pass extra options
  });

  // ✅ print here instead of inside sendRequest
  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// usage example
const response = await f2(mail["message"]);


<-- PROMPT --->
You are an intelligent assistant. Your task is to extract:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "category" as one of `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`

Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential Complex Name], where I have been residing for the past few years. I have always appreciated the meticulous care and attention your team provides in maintaining our facilities.

However, I am currently facing a pressing issue with the HVAC system in my apartment. Over the past f

## Sentiment & Urgency with Choices as `json`

In [15]:
// define the prompt
const prompt3 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "category" as one of {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above.

Giving the following message:`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

const f3 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt3,
    input: input,
    ...option_lists   // spreading urgency, sentiment, categories
  });

  // ✅ print outside sendRequest
  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// usage example
const response = await f3(mail["message"]);


<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "category" as one of `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above.

Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential Complex Name], where I have been residing for the past few years. I have always appreciated the meticulous c

## Sentiment & Urgency with Choices as `json` Improved Formatting

In [16]:
// define the prompt
const prompt4 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "category" as one of {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in \`\`\`json...\`\`\`, no newlines, no unnecessary whitespaces.;

Giving the following message:`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

// helper to replace placeholders {{?key}} with values from option_lists
function renderTemplate(messages, values) {
  return messages.map(msg => {
    let rendered = msg.content;
    for (const key in values) {
      rendered = rendered.replace(`{{?${key}}}`, values[key]);
    }
    return { ...msg, content: rendered };
  });
}
// equivalent to f_4 = partial(send_request, prompt=prompt_4, **option_lists)
const f4 = async (input) => {
  const renderedPrompt = renderTemplate(prompt4, option_lists);

  const { result, formattedPrompt } = await sendRequest({
    prompt: renderedPrompt,
    input: input
  });

  // ✅ handle printing here
  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// usage
const response = await f4(mail["message"]);


<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "category" as one of `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.;

Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential Complex Name], where I have bee

## Category Simple

In [17]:
// define the prompt
const prompt5 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Assign a list of matching support category to the message.

Giving the following message:`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

// equivalent to f_5 = partial(send_request, prompt=prompt_5)
const f5 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt5,
    input: input
  });

  // ✅ handle printing here, not inside sendRequest
  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// usage
const response = await f5(mail["message"]);


<-- PROMPT --->
You are an intelligent assistant. Assign a list of matching support category to the message.

Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential Complex Name], where I have been residing for the past few years. I have always appreciated the meticulous care and attention your team provides in maintaining our facilities.

However, I am currently facing a pressing issue with the HVAC system in my apartment. Over the past few days, the system has been malfunctioning, resulting in inconsistent temperatures and, at times, complete shutdowns. Given the current weather conditions, this has become quite unbearable and is affecting my daily routine significantly.

I have attempted to troubleshoot the problem by resetting the system and checking the thermostat settings, but these efforts have not yielded any improvement. The situation se

## Categories form list

In [18]:
const prompt6 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Assign a list best matching support category tags to the message:
{{?categories}}

Giving the following message:`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];
const f6 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt6,
    input: input,
    categories: option_lists.categories,   // ✅ only pass categories
  });

  // ✅ printing handled here
  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// usage
const response = await f6(mail["message"]);


<-- PROMPT --->
You are an intelligent assistant. Assign a list best matching support category tags to the message:
`facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`

Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential Complex Name], where I have been residing for the past few years. I have always appreciated the meticulous care and attention your team provides in maintaining our facilities.

However, I am currently facing a pressing issue with the HVAC system in my apartment. Over the past few days, the system has been malfunctioning, resulting in inconsistent temperatures an

## Category with  Descriptions and `json` Output

In [19]:
const prompt7 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Extract and return a json with the following keys and values:
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in \`\`\`json...\`\`\`, no newlines, no unnecessary whitespaces.

Giving the following message:`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

const f7 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt7,
    input: input,
    categories: option_lists.categories   // ✅ only categories, no urgency/sentiment
  });

  // ✅ print outside
  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// usage
const response = await f7(mail["message"]);


<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "categories" list of the best matching support category tags from: `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.

Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential Complex Name], where I have been residing for the past few years. I have always appreciate

## Complete

In [20]:
// Define the prompt
const prompt8 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned above. Never enclose it in \`\`\`json...\`\`\`, no newlines, no unnecessary whitespaces.
Giving the following message:`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

// Function to send request with the prompt and options
const f8 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt8,
    input: input,
    urgency: option_lists.urgency,
    sentiment: option_lists.sentiment,
    categories: option_lists.categories
  });

  // ✅ Log outside
  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// Usage
const response = await f8(mail["message"]);


<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "categories" list of the best matching support category tags from: `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.
Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential

# Evaluation

In [21]:
// Rate-limited iterator
class RateLimitedIterator {
  constructor(array, maxIterationsPerMinute) {
    this.array = array;
    this.index = 0;
    this.minInterval = 60 / maxIterationsPerMinute * 1000; // ms
    this.lastYieldTime = null;
  }

  async *[Symbol.asyncIterator]() {
    while (this.index < this.array.length) {
      const now = Date.now();
      if (this.lastYieldTime !== null) {
        const elapsed = now - this.lastYieldTime;
        if (elapsed < this.minInterval) {
          await new Promise(r => setTimeout(r, this.minInterval - elapsed));
        }
      }
      this.lastYieldTime = Date.now();
      yield this.array[this.index++];
    }
  }
}

Symbol(Symbol.asyncIterator)

In [22]:
// Evaluate a single mail
async function evaluation(mail, extractFunc, print = true, ...kwargs) {
  const response = await extractFunc(mail.message, { print, ...kwargs });

  let result = {
    is_valid_json: false,
    correct_categories: false,
    correct_sentiment: false,
    correct_urgency: false
  };

  let pred;
  try {
    pred = JSON.parse(response);
  } catch (e) {
    result.is_valid_json = false;
    return result;
  }

  result.is_valid_json = true;

  // categories: symmetric difference
  const mailCats = new Set(mail.ground_truth.categories);
  const predCats = new Set(pred.categories || []);
  const symDiff = new Set([...mailCats, ...predCats].filter(x => !mailCats.has(x) || !predCats.has(x)));
  result.correct_categories = 1 - (symDiff.size / mailCats.size);

  result.correct_sentiment = pred.sentiment === mail.ground_truth.sentiment;
  result.correct_urgency = pred.urgency === mail.ground_truth.urgency;

  return result;
}

In [23]:
// Evaluate a single mail
const result = await evaluation(mail, f8);
console.log(result);


<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "categories" list of the best matching support category tags from: `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.
Giving the following message:
Subject: Urgent HVAC System Repair Needed

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I am reaching out to you from [Residential

In [24]:
// Transpose list of objects to object of lists
function transposeListOfDicts(listOfDicts) {
  const keys = Object.keys(listOfDicts[0]);
  const transposed = {};
  for (const key of keys) transposed[key] = [];
  for (const obj of listOfDicts) {
    for (const key of keys) {
      transposed[key].push(obj[key]);
    }
  }
  return transposed;
}

// Evaluate full dataset
async function evaluationFullDataset(dataset, func, rateLimit = 100, print = false, ...kwargs) {
  const results = [];
  const iterator = new RateLimitedIterator(dataset, rateLimit);

  for await (const mail of iterator) {
    const res = await evaluation(mail, func, print, ...kwargs);
    results.push(res);
  }

  const transposed = transposeListOfDicts(results);

  // Compute average for each metric
  const n = dataset.length;
  for (const key in transposed) {
    transposed[key] = transposed[key].reduce((a, b) => a + b, 0) / n;
  }

  return transposed;
}

// Pretty print results
function prettyPrintTable(data) {
  const rowNames = Object.keys(data);
  if (!rowNames.length) return;

  const columnNames = Object.keys(data[rowNames[0]]);
  const rowNameWidth = Math.max(...rowNames.map(r => r.length));

  const columnWidths = columnNames.map(c => Math.max(c.length, ...rowNames.map(r => (data[r][c]*100).toFixed(1).length)));

  // Header
  let header = ' '.repeat(rowNameWidth) + ' ';
  header += columnNames.map((c, i) => c.padStart(columnWidths[i])).join(' ');
  console.log(header);
  console.log('='.repeat(header.length));

  // Rows
  for (const row of rowNames) {
    let line = row.padStart(rowNameWidth) + ' ';
    line += columnNames.map((c, i) => `${(data[row][c]*100).toFixed(1)}%`.padStart(columnWidths[i])).join(' ');
    console.log(line);
  }
}


In [25]:
// Initialize overallResult if not already defined
let overallResult = {};

// Append evaluation results for "basic--llama3-70b"
overallResult["basic--llama3.1-70b"] = await evaluationFullDataset(test_set_small, f8);

// Show the updated results table
prettyPrintTable(overallResult);

<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "categories" list of the best matching support category tags from: `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.
Giving the following message:
Subject: Inquiry About Specialized Cleaning Services

Dear Facility Solutions Company Support Team,

I hope this message finds you well. My name is [Sender], and I am

Rerunning the cell above shows that the output depends a lot on the given input example

## Few-Shot Example

In [26]:
import seedrandom from "seedrandom";

// prompt definition
const prompt10 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Your task is to extract and categorize messages. Here are some example:
{{?few_shot_examples}}
Use the examples when extract and categorize the following message:
Extract and return a json with the follwoing keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in \`\`\`json...\`\`\`, no newlines, no unnecessary whitespaces.`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

// reproducible random generator
const rng = seedrandom("42");

// helper: random sample from array
function randomSample(array, k, rng) {
  const result = [];
  const arrCopy = [...array];
  for (let i = 0; i < k && arrCopy.length > 0; i++) {
    const idx = Math.floor(rng() * arrCopy.length);
    result.push(arrCopy[idx]);
    arrCopy.splice(idx, 1);
  }
  return result;
}

// pick k = 3 examples
const k = 3;
const examplesSample = randomSample(dev_set, k, rng);

// example template
const exampleTemplate = (example) => `<example>
${example.message}

## Output
${JSON.stringify(example.ground_truth)}
</example>`;

// join examples
const examples = examplesSample.map(exampleTemplate).join("\n---\n");

// function wrapper
const f10 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt10,
    few_shot_examples: examples,
    ...option_lists,
    input: input
  });

  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);

  return result;
};

// usage example
const response = await f10(mail["message"]);


<-- PROMPT --->
You are an intelligent assistant. Your task is to extract and categorize messages. Here are some example:
<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followe

In [27]:
// Evaluate with few-shot prompt (f10) and add results to overallResult
overallResult["fewshot--llama3.1-70b"] = await evaluationFullDataset(test_set_small, f10);

// Print updated results table with percentages
prettyPrintTable(overallResult);


<-- PROMPT --->
You are an intelligent assistant. Your task is to extract and categorize messages. Here are some example:
<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followe

## Use Metaprompting

In [28]:
// Example template for metaprompting
const exampleTemplateMetaprompt = (exampleInput, key, exampleOutput) => `
<example>
${exampleInput}

## Output
${key}=${exampleOutput}
</example>`;

// Prompt for guide generation
const promptGetGuide = [
  {
    role: "system",
    content: `You are an intelligent assistant. Here are some examples:
---
{{?examples}}
---
Use the examples above to come up with a guide on how to distinguish between {{?options}} {{?key}}.

Format the guide like this:

### **<category 1>**
- <instruction 1>
- <instruction 2>
- <instruction 3>

### **<category 2>**
- <instruction 1>
- <instruction 2>
- <instruction 3>

...

When creating the guide:
- Write step-by-step instructions
- Consider that some labels in the examples might be incorrect
- Avoid copying explicit details from the examples
- Ensure the guide covers: {{?options}}`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

// Store generated guides
const guides = {};

// Loop over the keys and generate guides
for (const key of ["categories", "urgency", "sentiment"]) {
  const options = option_lists[key];

  // Build examples string
  const selectedExamplesTxtMetaprompt = dev_set
    .map(
      (example) =>
        exampleTemplateMetaprompt(
          example.message,
          key,
          example.ground_truth[key]
        )
    )
    .join("\n---\n");

  // Call LLM
  const { result, formattedPrompt } = await sendRequest({
    prompt: promptGetGuide,
    examples: selectedExamplesTxtMetaprompt,
    input: selectedExamplesTxtMetaprompt,
    key: key,
    options: options
  });

  console.log(`\n<-- GUIDE PROMPT for ${key} -->\n${formattedPrompt}`);
  console.log(`<-- RESPONSE for ${key} -->\n${result}\n`);

  guides[`guide_${key}`] = result;
}



<-- GUIDE PROMPT for categories -->
You are an intelligent assistant. Here are some examples:
---

<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followed, causing considerabl

"### **Neutral Sentiment**\n" +
  "\n" +
  "- The message is primarily informative, asking for details about services, schedules, or procedures.\n" +
  "- The tone is polite and inquiring, without expressing strong emotions or opinions.\n" +
  "- The language used is formal and professional, indicating a neutral or objective tone.\n" +
  "- The message may contain some minor concerns or issues, but they are presented in a non-confrontational manner.\n" +
  "- The overall tone is calm and cooperative, seeking assistance or guidance without being overly critical or negative.\n" +
  "\n" +
  "### **Positive Sentiment**\n" +
  "\n" +
  "- The message expresses appreciation or gratitude for the services provided.\n" +
  "- The tone is friendly and enthusiastic, indicating a positive experience or interaction.\n" +
  "- The language used is warm and complimentary, highlighting the benefits or value of the services.\n" +
  "- The message may contain some minor issues or concerns, but they are

In [29]:
  // print guide for urgency
  console.log(guides["guide_urgency"]);

### **High Urgency**

- The issue is critical and requires immediate attention.
- The problem is causing significant disruptions or safety concerns.
- The customer is experiencing extreme discomfort or distress.
- The issue is time-sensitive and requires prompt resolution.
- The customer explicitly states that the issue is urgent or requires immediate attention.

### **Medium Urgency**

- The issue is important but not critical.
- The problem is causing some disruptions or inconvenience.
- The customer is experiencing moderate discomfort or frustration.
- The issue requires attention within a reasonable timeframe.
- The customer implies that the issue needs to be addressed soon.

### **Low Urgency**

- The issue is minor or routine.
- The problem is not causing significant disruptions or inconvenience.
- The customer is not experiencing significant discomfort or frustration.
- The issue can be addressed at a convenient time.
- The customer is seeking information or guidance rather than

In [30]:
// prompt definition
const prompt12 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Your task is to classify messages.
This is an explanation of \`urgency\` labels:
---
{{?guide_urgency}}
---
This is an explanation of \`sentiment\` labels:
---
{{?guide_sentiment}}
---
This is an explanation of \`support\` categories:
---
{{?guide_categories}}
---
Giving the following message:

Extract and return a json with the following keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in \`\`\`json...\`\`\`, no newlines, no unnecessary whitespaces.`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

// wrapper function for prompt12
const f12 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt12,
    input:input,
    ...option_lists,
    ...guides,
  });
    
  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);
  return result;
};

// usage example
const response = await f12(mail.message);


<-- PROMPT --->
You are an intelligent assistant. Your task is to classify messages.
This is an explanation of `urgency` labels:
---
### **High Urgency**

- The issue is critical and requires immediate attention.
- The problem is causing significant disruptions or safety concerns.
- The customer is experiencing extreme discomfort or distress.
- The issue is time-sensitive and requires prompt resolution.
- The customer explicitly states that the issue is urgent or requires immediate attention.

### **Medium Urgency**

- The issue is important but not critical.
- The problem is causing some disruptions or inconvenience.
- The customer is experiencing moderate discomfort or frustration.
- The issue requires attention within a reasonable timeframe.
- The customer implies that the issue needs to be addressed soon.

### **Low Urgency**

- The issue is minor or routine.
- The problem is not causing significant disruptions or inconvenience.
- The customer is not experiencing significant discom

In [31]:
overallResult["metadata--llama3.1-70b"] = await evaluationFullDataset(test_set_small, f12);

// Print updated results table with percentages
prettyPrintTable(overallResult);

<-- PROMPT --->
You are an intelligent assistant. Your task is to classify messages.
This is an explanation of `urgency` labels:
---
### **High Urgency**

- The issue is critical and requires immediate attention.
- The problem is causing significant disruptions or safety concerns.
- The customer is experiencing extreme discomfort or distress.
- The issue is time-sensitive and requires prompt resolution.
- The customer explicitly states that the issue is urgent or requires immediate attention.

### **Medium Urgency**

- The issue is important but not critical.
- The problem is causing some disruptions or inconvenience.
- The customer is experiencing moderate discomfort or frustration.
- The issue requires attention within a reasonable timeframe.
- The customer implies that the issue needs to be addressed soon.

### **Low Urgency**

- The issue is minor or routine.
- The problem is not causing significant disruptions or inconvenience.
- The customer is not experiencing significant discom

## Combine Few-Shot with Metaprompting

In [32]:
// prompt definition
const prompt13 = [
  {
    role: "system",
    content: `You are an intelligent assistant. Your task is to classify messages.
Here are some examples:
---
{{?few_shot_examples}}
---
This is an explanation of \`urgency\` labels:
---
{{?guide_urgency}}
---
This is an explanation of \`sentiment\` labels:
---
{{?guide_sentiment}}
---
This is an explanation of \`support\` categories:
---
{{?guide_categories}}
---
Giving the following message:

extract and return a json with the following keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in \`\`\`json...\`\`\`, no newlines, no unnecessary whitespaces.`
  },
  {
    role: "user",
    content:'{{?input}}'
  }
];

// wrapper function for prompt13
const f13 = async (input) => {
  const { result, formattedPrompt } = await sendRequest({
    prompt: prompt13,
    input,   
    ...option_lists,
    few_shot_examples: examples,
    ...guides,
  });

  console.log("<-- PROMPT --->\n" + formattedPrompt + "\n<--- RESPONSE --->\n" + result);
  return result;
};

// usage example
const response = await f13(mail.message);


<-- PROMPT --->
You are an intelligent assistant. Your task is to classify messages.
Here are some examples:
---
<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followed, causin

In [33]:
overallResult["metaprompting_and_few_shot--llama3.1-70b"] = await evaluationFullDataset(test_set_small, f13);

// Print updated results table with percentages
prettyPrintTable(overallResult);

<-- PROMPT --->
You are an intelligent assistant. Your task is to classify messages.
Here are some examples:
---
<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followed, causin

## Use `mistralai--mistral-large-instruct`

As the cheapest open source, SAP hosted model available on generative AI hub

In [34]:
overallResult["basic--mistral-large-instruct"] = await evaluationFullDataset(test_set_small, f8, { _model: "mistralai--mistral-large-instruct" });

<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "categories" list of the best matching support category tags from: `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.
Giving the following message:
Subject: Inquiry About Specialized Cleaning Services

Dear Facility Solutions Company Support Team,

I hope this message finds you well. My name is [Sender], and I am

{
  is_valid_json: 1,
  correct_categories: 0.2425,
  correct_sentiment: 0.3,
  correct_urgency: 0.75
}

In [35]:
// Print updated results table with percentages
prettyPrintTable(overallResult);

                                         is_valid_json correct_categories correct_sentiment correct_urgency
                     basic--llama3.1-70b        100.0%              22.8%             40.0%           80.0%
                   fewshot--llama3.1-70b        100.0%              30.0%             50.0%           85.0%
                  metadata--llama3.1-70b        100.0%              50.6%             50.0%           90.0%
metaprompting_and_few_shot--llama3.1-70b        100.0%              31.8%             70.0%           85.0%
           basic--mistral-large-instruct        100.0%              24.3%             30.0%           75.0%


In [36]:
overallResult["metaprompting_and_few_shot--mixtral-large-instruct"] = await evaluationFullDataset(test_set_small, f13, { _model: "mistralai--mistral-large-instruct" });

<-- PROMPT --->
You are an intelligent assistant. Your task is to classify messages.
Here are some examples:
---
<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followed, causin

{
  is_valid_json: 1,
  correct_categories: 0.3875,
  correct_sentiment: 0.65,
  correct_urgency: 0.75
}

In [37]:
// Print updated results table with percentages
prettyPrintTable(overallResult);

                                                   is_valid_json correct_categories correct_sentiment correct_urgency
                               basic--llama3.1-70b        100.0%              22.8%             40.0%           80.0%
                             fewshot--llama3.1-70b        100.0%              30.0%             50.0%           85.0%
                            metadata--llama3.1-70b        100.0%              50.6%             50.0%           90.0%
          metaprompting_and_few_shot--llama3.1-70b        100.0%              31.8%             70.0%           85.0%
                     basic--mistral-large-instruct        100.0%              24.3%             30.0%           75.0%
metaprompting_and_few_shot--mixtral-large-instruct        100.0%              38.8%             65.0%           75.0%


## Use `gpt-4o`

As the best propriatary OpenAI available on generative AI hub

In [38]:
overallResult["basic--gpt4o"] = await evaluationFullDataset(test_set_small, f8, { _model: "gpt-4o" });

<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "categories" list of the best matching support category tags from: `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.
Giving the following message:
Subject: Inquiry About Specialized Cleaning Services

Dear Facility Solutions Company Support Team,

I hope this message finds you well. My name is [Sender], and I am

{
  is_valid_json: 1,
  correct_categories: 0.2325,
  correct_sentiment: 0.5,
  correct_urgency: 0.7
}

In [39]:
// Print updated results table with percentages
prettyPrintTable(overallResult);

                                                   is_valid_json correct_categories correct_sentiment correct_urgency
                               basic--llama3.1-70b        100.0%              22.8%             40.0%           80.0%
                             fewshot--llama3.1-70b        100.0%              30.0%             50.0%           85.0%
                            metadata--llama3.1-70b        100.0%              50.6%             50.0%           90.0%
          metaprompting_and_few_shot--llama3.1-70b        100.0%              31.8%             70.0%           85.0%
                     basic--mistral-large-instruct        100.0%              24.3%             30.0%           75.0%
metaprompting_and_few_shot--mixtral-large-instruct        100.0%              38.8%             65.0%           75.0%
                                      basic--gpt4o        100.0%              23.3%             50.0%           70.0%


In [40]:
overallResult["metaprompting_and_few_shot--gpt4o"] = await evaluationFullDataset(test_set_small, f13, { _model: "gpt-4o" });

<-- PROMPT --->
You are an intelligent assistant. Your task is to classify messages.
Here are some examples:
---
<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followed, causin

{
  is_valid_json: 1,
  correct_categories: 0.37083333333333335,
  correct_sentiment: 0.6,
  correct_urgency: 0.8
}

In [41]:
// Print updated results table with percentages
prettyPrintTable(overallResult);

                                                   is_valid_json correct_categories correct_sentiment correct_urgency
                               basic--llama3.1-70b        100.0%              22.8%             40.0%           80.0%
                             fewshot--llama3.1-70b        100.0%              30.0%             50.0%           85.0%
                            metadata--llama3.1-70b        100.0%              50.6%             50.0%           90.0%
          metaprompting_and_few_shot--llama3.1-70b        100.0%              31.8%             70.0%           85.0%
                     basic--mistral-large-instruct        100.0%              24.3%             30.0%           75.0%
metaprompting_and_few_shot--mixtral-large-instruct        100.0%              38.8%             65.0%           75.0%
                                      basic--gpt4o        100.0%              23.3%             50.0%           70.0%
                 metaprompting_and_few_shot--gpt4o      

## Using `gemini-2.5-flash`

As the cheapest and fastest Google model available on generative AI hub

In [42]:
overallResult["basic--gemini-2.5-flash"] = await evaluationFullDataset(test_set_small, f8, { _model: "gemini-2.5-flash" });

<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "categories" list of the best matching support category tags from: `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.
Giving the following message:
Subject: Inquiry About Specialized Cleaning Services

Dear Facility Solutions Company Support Team,

I hope this message finds you well. My name is [Sender], and I am

{
  is_valid_json: 1,
  correct_categories: 0.3375,
  correct_sentiment: 0.4,
  correct_urgency: 0.75
}

In [43]:
// Print updated results table with percentages
prettyPrintTable(overallResult);

                                                   is_valid_json correct_categories correct_sentiment correct_urgency
                               basic--llama3.1-70b        100.0%              22.8%             40.0%           80.0%
                             fewshot--llama3.1-70b        100.0%              30.0%             50.0%           85.0%
                            metadata--llama3.1-70b        100.0%              50.6%             50.0%           90.0%
          metaprompting_and_few_shot--llama3.1-70b        100.0%              31.8%             70.0%           85.0%
                     basic--mistral-large-instruct        100.0%              24.3%             30.0%           75.0%
metaprompting_and_few_shot--mixtral-large-instruct        100.0%              38.8%             65.0%           75.0%
                                      basic--gpt4o        100.0%              23.3%             50.0%           70.0%
                 metaprompting_and_few_shot--gpt4o      

In [44]:
overallResult["metaprompting_and_few_shot--gemini-2.5-flash"] = await evaluationFullDataset(test_set_small, f13, { _model: "gemini-2.5-flash" });

<-- PROMPT --->
You are an intelligent assistant. Your task is to classify messages.
Here are some examples:
---
<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followed, causin

{
  is_valid_json: 1,
  correct_categories: 0.32083333333333336,
  correct_sentiment: 0.7,
  correct_urgency: 0.8
}

In [45]:
// Print updated results table with percentages
prettyPrintTable(overallResult);

                                                   is_valid_json correct_categories correct_sentiment correct_urgency
                               basic--llama3.1-70b        100.0%              22.8%             40.0%           80.0%
                             fewshot--llama3.1-70b        100.0%              30.0%             50.0%           85.0%
                            metadata--llama3.1-70b        100.0%              50.6%             50.0%           90.0%
          metaprompting_and_few_shot--llama3.1-70b        100.0%              31.8%             70.0%           85.0%
                     basic--mistral-large-instruct        100.0%              24.3%             30.0%           75.0%
metaprompting_and_few_shot--mixtral-large-instruct        100.0%              38.8%             65.0%           75.0%
                                      basic--gpt4o        100.0%              23.3%             50.0%           70.0%
                 metaprompting_and_few_shot--gpt4o      

## Use `gemini-2.5-pro`

Best Google model available on generative AI hub

In [46]:
overallResult["basic--gemini-2.5-pro"] = await evaluationFullDataset(test_set_small, f8, { _model: "gemini-2.5-pro" });

<-- PROMPT --->
You are an intelligent assistant. Extract and return a json with the following keys and values:
- "urgency" as one of `high`, `medium`, `low`
- "sentiment" as one of `neutral`, `positive`, `negative`
- "categories" list of the best matching support category tags from: `facility_management_issues`, `quality_and_safety_concerns`, `emergency_repair_services`, `training_and_support_requests`, `routine_maintenance_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `general_inquiries`, `specialized_cleaning_services`, `sustainability_and_environmental_practices`
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned above. Never enclose it in ```json...```, no newlines, no unnecessary whitespaces.
Giving the following message:
Subject: Inquiry About Specialized Cleaning Services

Dear Facility Solutions Company Support Team,

I hope this message finds you well. My name is [Sender], and I am

{
  is_valid_json: 1,
  correct_categories: 0.22833333333333336,
  correct_sentiment: 0.45,
  correct_urgency: 0.75
}

In [47]:
// Print updated results table with percentages
prettyPrintTable(overallResult);

                                                   is_valid_json correct_categories correct_sentiment correct_urgency
                               basic--llama3.1-70b        100.0%              22.8%             40.0%           80.0%
                             fewshot--llama3.1-70b        100.0%              30.0%             50.0%           85.0%
                            metadata--llama3.1-70b        100.0%              50.6%             50.0%           90.0%
          metaprompting_and_few_shot--llama3.1-70b        100.0%              31.8%             70.0%           85.0%
                     basic--mistral-large-instruct        100.0%              24.3%             30.0%           75.0%
metaprompting_and_few_shot--mixtral-large-instruct        100.0%              38.8%             65.0%           75.0%
                                      basic--gpt4o        100.0%              23.3%             50.0%           70.0%
                 metaprompting_and_few_shot--gpt4o      

In [48]:
overallResult["metaprompting_and_few_shot--gemini-2.5-pro"] = await evaluationFullDataset(test_set_small, f13, { _model: "gemini-2.5-pro" });

<-- PROMPT --->
You are an intelligent assistant. Your task is to classify messages.
Here are some examples:
---
<example>
Subject: Urgent Assistance Required for Facility Management and Safety Concerns

Dear Support Team,

I hope this message finds you well. My name is [Sender], and I have been a client of Facility Solutions Company for the past two years, primarily utilizing your facility management services for our residential complex in Toronto. I am reaching out to seek your immediate assistance with a matter that has recently come to my attention.

Over the past few weeks, we have noticed some serious inconsistencies in the management of our facility operations. Specifically, there have been significant lapses in the coordination of space utilization and security measures. For instance, the common areas are not being utilized efficiently, leading to overcrowding during peak hours. Additionally, there have been a few instances where the security protocols were not followed, causin

{
  is_valid_json: 1,
  correct_categories: 0.3458333333333333,
  correct_sentiment: 0.65,
  correct_urgency: 0.8
}

In [49]:
// Print updated results table with percentages
prettyPrintTable(overallResult);

                                                   is_valid_json correct_categories correct_sentiment correct_urgency
                               basic--llama3.1-70b        100.0%              22.8%             40.0%           80.0%
                             fewshot--llama3.1-70b        100.0%              30.0%             50.0%           85.0%
                            metadata--llama3.1-70b        100.0%              50.6%             50.0%           90.0%
          metaprompting_and_few_shot--llama3.1-70b        100.0%              31.8%             70.0%           85.0%
                     basic--mistral-large-instruct        100.0%              24.3%             30.0%           75.0%
metaprompting_and_few_shot--mixtral-large-instruct        100.0%              38.8%             65.0%           75.0%
                                      basic--gpt4o        100.0%              23.3%             50.0%           70.0%
                 metaprompting_and_few_shot--gpt4o      